In [1]:
!pip install "unsloth @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install kagglehub datasets transformers
!pip install unsloth_zoo

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-0j2gwg11/unsloth_87764a64d7774bd0884632d802f15cfe
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-0j2gwg11/unsloth_87764a64d7774bd0884632d802f15cfe
  Resolved https://github.com/unslothai/unsloth.git to commit ab9689c034ace6fef573ce017bbb30908826b114
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.6.2-py3-none-any.whl size=35629643 sha256=2d4661942de6a769726179def0ab9f0851bf2c8e4a604d00c86c3d982d9aed7d
  Stored in directory: /tmp/pip-ephem-wheel-cache-ljk73b1k/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 45.7 MB/s eta 0:00:00
   ━━━━━━

In [2]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

import kagglehub
import os
import json
import glob
from datasets import Dataset
from transformers import EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
def load_and_format_dataset(file_path):
    formatted_data = []
    line_count = 0
    skipped_count = 0

    with open(file_path, 'r', encoding='utf-8') as f:
        first_char = f.read(1)
        f.seek(0)

        if first_char == '[':
            records = json.load(f)
        else:
            records = []
            for line in f:
                line_count += 1
                cleaned_line = line.strip()
                if not cleaned_line: continue
                try:
                    records.append(json.loads(cleaned_line))
                except json.JSONDecodeError:
                    skipped_count += 1
                    continue

    for record in records:
        jd = record.get("job_description", "")
        questions = record.get("evaluation_questions", "")
        if not jd or not questions: continue

        messages = [
            {"role": "system", "content": "You are a professional HR assistant. Analyze the provided Job Description and generate categorized resume evaluation questions."},
            {"role": "user", "content": f"Generate evaluation questions for this job description:\n\n{jd}"},
            {"role": "assistant", "content": questions}
        ]
        formatted_data.append({"messages": messages})

    print(f"Successfully loaded {len(formatted_data)} records. (Skipped: {skipped_count} rows)")
    return Dataset.from_list(formatted_data)

def apply_formatting_template(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset_dir = "/kaggle/input/datasets/adhamashraf202200953/techncial-generated-job-descriptions"
print(f'Data source import complete. Directory path: {dataset_dir}')

Data source import complete. Directory path: /kaggle/input/datasets/adhamashraf202200953/techncial-generated-job-descriptions


In [4]:
INPUT_FILE = "/kaggle/input/datasets/adhamashraf202200953/techncial-generated-job-descriptions/phase2_questions.jsonl"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
OUTPUT_DIR = "/kaggle/working/outputs/jd_to_questions_model"

MAX_SEQ_LENGTH = 2048 
LOAD_IN_4BIT = True

raw_dataset = load_and_format_dataset(INPUT_FILE)
dataset_split = raw_dataset.train_test_split(test_size=0.05, seed=3407)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
)

template_type = "llama-3.1" if "llama" in MODEL_NAME.lower() else ("qwen-2.5" if "qwen" in MODEL_NAME.lower() else "phi-3.5")
tokenizer = get_chat_template(tokenizer, chat_template = template_type)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = train_dataset.map(apply_formatting_template, batched = True)
eval_dataset = eval_dataset.map(apply_formatting_template, batched = True)

Successfully loaded 5014 records. (Skipped: 1 rows)
==((====))==  Unsloth 2026.6.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/4763 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

In [5]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    dataset_text_field = "text",     
    max_length = MAX_SEQ_LENGTH, 
    per_device_train_batch_size = 1,       
    gradient_accumulation_steps = 8,       
    per_device_eval_batch_size = 1,
    eval_accumulation_steps = 1,
    warmup_steps = 10,
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = True,                          
    logging_steps = 10,
    eval_strategy = "steps",
    eval_steps = 100,
    save_strategy = "steps",
    save_steps = 100,                      
    save_total_limit = 2,                 
    load_best_model_at_end = False,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    optim = "adamw_8bit",                  
    weight_decay = 0.01,
    seed = 3407,
    output_dir = "/kaggle/working/outputs",
)

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = args,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 2)]
)

checkpoint_dirs = glob.glob("/kaggle/working/outputs/checkpoint-*")
resume_checkpoint = False

if checkpoint_dirs:
    latest_checkpoint = max(checkpoint_dirs, key=os.path.getctime)
    print(f"Found active checkpoint directory. Resuming training from: {latest_checkpoint}")
    resume_checkpoint = latest_checkpoint
else:
    print("No historical checkpoints detected. Initializing a clean training session...")

trainer_stats = trainer.train(resume_from_checkpoint=resume_checkpoint)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/4763 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/251 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
No historical checkpoints detected. Initializing a clean training session...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,763 | Num Epochs = 1 | Total steps = 596
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
100,0.555988,0.541649
200,0.509775,0.514436
300,0.490675,0.501851
400,0.479333,0.491708
500,0.481410,0.484833
596,0.471179,0.481673


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/outputs/checkpoint-596/tokenizer_config.json.


In [6]:
best_checkpoint_path = trainer.state.best_model_checkpoint

if best_checkpoint_path is not None:
    print(f"Loading best checkpoint from: {best_checkpoint_path}")
    model.from_pretrained(model, best_checkpoint_path)
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
else:
    print(f"Saving final trained base state to: {OUTPUT_DIR}")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

!zip -r "qwen_qag.zip" "/kaggle/working/outputs"

Loading best checkpoint from: /kaggle/working/outputs/checkpoint-596


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.

  adding: kaggle/working/outputs/ (stored 0%)
  adding: kaggle/working/outputs/checkpoint-596/ (stored 0%)
  adding: kaggle/working/outputs/checkpoint-596/scheduler.pt (deflated 62%)
  adding: kaggle/working/outputs/checkpoint-596/tokenizer_config.json (deflated 89%)
  adding: kaggle/working/outputs/checkpoint-596/trainer_state.json (deflated 75%)
  adding: kaggle/working/outputs/checkpoint-596/README.md (deflated 65%)
  adding: kaggle/working/outputs/checkpoint-596/optimizer.pt (deflated 12%)
  adding: kaggle/working/outputs/checkpoint-596/rng_state.pth (deflated 26%)
  adding: kaggle/working/outputs/checkpoint-596/tokenizer.json (deflated 81%)
  adding: kaggle/working/outputs/checkpoint-596/training_args.bin (deflated 53%)
  adding: kaggle/working/outputs/checkpoint-596/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/outputs/checkpoint-596/adapter_config.json (deflated 57%)
  adding: kaggle/working/outputs/checkpoint-596/chat_template.jinja (deflated 71%)
  adding: k

In [7]:
print("\n--- Starting Local Inference Test ---\n")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = OUTPUT_DIR,
    max_seq_length = 4096,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

test_jd = """
We are looking for a Senior DevOps Engineer with 5+ years of experience.
Must have strong expertise in Kubernetes cluster administration, Terraform for AWS Infrastructure,
and building Gitlab CI/CD pipelines. Experience with Prometheus and Grafana is a plus.
"""

messages = [
    {
        "role": "system",
        "content": (
            "You are an automated HR Screening Bot. Your job is to analyze the Job Description and generate "
            "objective verification check questions for an LLM to evaluate a Candidate's CV.\n\n"
            "CRITICAL RULES:\n"
            "1. DO NOT use second-person pronouns ('you', 'your', 'yours').\n"
            "2. DO NOT frame questions as a direct interview interview ('What is your experience...').\n"
            "3. Frame questions as a third-person verification query ('Does the CV show experience with...', "
            "'Is there evidence of the candidate managing...', 'Verify if the candidate has worked on...').\n"
            "4. Maintain strict markdown headers."
        )
    },
    {
        "role": "user",
        "content": f"Generate CV evaluation verification questions for this job description:\n\n{test_jd}"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict = True,
).to("cuda")

outputs = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 1024,
    use_cache = True,
    temperature = 0.5,
    top_p = 0.9
)

print(tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True))


--- Starting Local Inference Test ---

==((====))==  Unsloth 2026.6.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


# Does the CV show experience with administering Kubernetes clusters for at least 5 years?
# Is there evidence of the candidate using Terraform for AWS infrastructure management for more than 5 years?
# Verify if the candidate has built Gitlab CI/CD pipelines as part of their work experience.
# Has the candidate demonstrated experience with Prometheus monitoring tools on their CV?
# Is there any mention of the candidate's involvement with Grafana dashboard creation or management?
